# P132 — Splatting de gaussianas 3D para renderizado de campos de radiancia en tiempo real

## 1. Título y paper

**Paper:** *3D Gaussian Splatting for Real-Time Radiance Field Rendering*  
**Autoría:** Bernhard Kerbl, Georgios Kopanas, Thomas Leimkühler, George Drettakis  
**Año y venue:** 2023 · ACM Transactions on Graphics, 42(4)  
**Nivel:** L3 · **Motor:** `gaussian_splatting`  
**Ficha completa:** [`P132_gaussian_splatting`](../../papers/foundational/P132_gaussian_splatting/README.md)

**Hito:** Alcanza calidad de campo de radiancia a velocidad de tiempo real cambiando la función continua por millones de primitivas explícitas que se rasterizan.

- [doi:10.1145/3592433](https://doi.org/10.1145/3592433)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: NeRF produce vistas excelentes y renderiza lentísimo: cada píxel exige decenas de consultas a un perceptrón a lo largo de su rayo, y la mayoría caen en el vacío. Eso lo deja fuera de cualquier aplicación interactiva.
2. Ejecutar una implementación mínima de la propuesta: Representar la escena como un conjunto de gaussianas 3D anisótropas con color y opacidad, optimizadas desde las vistas de entrada, y renderizarlas proyectándolas y mezclándolas por orden de profundidad con un rasterizador diseñado a medida.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P128


## 4. Intuición

NeRF gasta la mayor parte de su cómputo **muestreando el vacío**: con una escena ocupada al 1 %, 190 de las 192 muestras de cada rayo caen donde no hay nada.


## 5. Concepto mínimo

```text
NeRF      : por píxel, N muestras × una pasada del perceptrón por muestra
Splatting : proyectar las gaussianas y mezclar solo las que caen en el píxel

Se cambia CÓMPUTO por MEMORIA
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gaussian_splatting', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántas muestras por rayo se desperdician?
2. ¿Cuánto cuesta cada método por fotograma?
3. ¿Qué se paga a cambio?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gaussian_splatting', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('gaussian_splatting', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con la escena ocupada al 1 %, **190,1 de 192** muestras por rayo caen en el vacío. El conteo bruto da 2,09e+14 operaciones frente a 1,2e+09 — una razón enorme que **no** es la aceleración real: el artículo mide del orden de **mil veces**. Y el precio es memoria: **2,1 MB** el perceptrón contra **236 MB** el millón de gaussianas.


## 10. Comentario pedagógico

Fíjate en la honestidad de esa comparación: el conteo de operaciones exagera cinco órdenes de magnitud porque ignora el muestreo jerárquico de NeRF y cómo aprovecha cada método la GPU. Un número calculado no sustituye a uno medido, y esta es exactamente la clase de cifra que se cita mal.


## 11. Error o anti-patrón deliberado

Anti-patrón: comparar arquitecturas contando operaciones.


In [ ]:
print('El conteo bruto da 175 000x. La medicion del articulo da ~1000x.')
print('La diferencia es toda de ingenieria: ocupacion de la GPU, jerarquia, memoria.')
print('Si tu comparacion no esta medida, no es una comparacion.')

## 12. Corrección

Dónde se va el cómputo y dónde la memoria:


In [ ]:
r = run_paper_lab('gaussian_splatting', seed=3)['result']
print('nerf:', r['nerf'])
print('splatting:', r['splatting'])
print('memoria:', r['memoria'])
print('muestreo del vacio:')
for f in r['muestreo_del_vacio']:
    print('  ', f)

## 13. Desafío guiado

Explica qué gana una representación explícita más allá de la velocidad, y por qué eso importa para un flujo de trabajo de producción.


In [ ]:
r = run_paper_lab('gaussian_splatting', seed=3)['result']
show(r)

## 14. Desafío autónomo

Estima cuánta memoria ocuparía una escena tuya como gaussianas y decide si cabe en el dispositivo donde tendría que renderizarse.


## 15. Evidencia de aprendizaje

Guarda la estimación y la conclusión.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P132_gaussian_splatting/README.md) · evaluación formal: [`assessments/papers/P132_gaussian_splatting.md`](../../assessments/papers/P132_gaussian_splatting.md)


## 16. Cierre

Queda la consecuencia que ninguna de estas técnicas evita: lo generado acaba en el corpus del modelo siguiente.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
